# Imports

In [ ]:
import pandas as pd
import os
import json

# 1. Path Setup for Docker/Local consistency
current_dir = os.getcwd()
if os.path.basename(current_dir) == 'ETL':
    project_root = os.path.abspath(os.path.join(current_dir, '..'))
else:
    project_root = current_dir

if project_root not in sys.path:
    sys.path.append(project_root)

# 2. Ensure working directory is correct for data loading
if os.path.basename(os.getcwd()) != 'ETL' and os.path.exists('ETL'):
    os.chdir('ETL')

print(f"Working Directory: {os.getcwd()}")

# Configurations

In [ ]:
# Define relative paths 
INPUT_FILE_POSTS = "../data_temp/blog_posts_1_intermediate.json"
INPUT_FILE_MEMBERS = "../data_raw/wix_members_data.json"
OUTPUT_FOLDER = "../data_processed/"
OUTPUT_FILE = os.path.join(OUTPUT_FOLDER, "blog_posts.json")

# Simple check to verify the files is where we think they are
if os.path.exists(INPUT_FILE_POSTS ):
    print(f"✅ Setup complete. Input files found: {INPUT_FILE_POSTS}")
else:
    print(f"❌ Warning: Input file NOT found at {INPUT_FILE_POSTS }")

if os.path.exists(INPUT_FILE_MEMBERS ):
    print(f"✅ Setup complete. Input files found: {INPUT_FILE_MEMBERS}")
else:
    print(f"❌ Warning: Input file NOT found at {INPUT_FILE_MEMBERS }")

# Ensure the output directory exists
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)
    print(f"Created folder: {OUTPUT_FOLDER}")

# Load raw data

In [ ]:
try:
    posts_df = pd.read_json(INPUT_FILE_POSTS)
    print(f"✅ Successfully loaded {len(posts_df)} records.")
    
    display(posts_df.head(3)) 
    
    print("\nAvailable columns:", *posts_df.columns, sep="\n")
  
except FileNotFoundError:
    print(f"Error: The file {INPUT_FILE_POSTS} was not found.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
with open(INPUT_FILE_MEMBERS, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# Extract only the list from "_items" and create the DataFrame
# This keeps the nested 'profile' as a dictionary inside the cell
members_df = pd.DataFrame(raw_data["_items"])

print(f"✅ Successfully loaded {len(members_df)} member records.")
display(members_df.head(3))

# Pair unique Member IDs from `posts_df` with Names from `members_df`

In [ ]:
# Extract unique member IDs and create a new DataFrame
unique_member_ids_df = pd.DataFrame(posts_df['memberId'].unique(),
columns=['memberId'])

# Display the resulting DataFrame
print(f"Found {len(unique_member_ids_df)} unique member IDs.")
display(unique_member_ids_df)

In [ ]:
# 1. Extract 'nickname' from the 'profile' dictionary in members_df
members_df['name'] = members_df['profile'].apply(lambda x: 
x.get('nickname') if isinstance(x, dict) else None)

# 2. Merge the nickname into unique_member_ids_df
# We match 'memberId' from unique_member_ids_df with '_id' from members_df
unique_member_ids_df = unique_member_ids_df.merge(
    members_df[['_id', 'name']],
    left_on='memberId',
    right_on='_id',
    how='left'
).drop(columns=['_id']) # Remove the redundant '_id' column after merge

# Display the result
display(unique_member_ids_df)

# Map Member IDs to Human-Readable names inside `posts_df`

In [ ]:
# 1. Create a mapping dictionary from unique_member_ids_df
id_to_name = dict(zip(unique_member_ids_df['memberId'],
unique_member_ids_df['name']))

# 2. Update the 'author' column and drop the 'memberId' column
posts_df['author'] = posts_df['memberId'].map(id_to_name)
posts_df = posts_df.drop(columns=['memberId'])

# Display the result to verify
print("Updated blog posts with author names:")
display(posts_df[['title', 'author']].head())

# Export to JSON 

In [ ]:
# Keeping Hungarian characters safe with ensure_ascii=False
posts_df.to_json(OUTPUT_FILE, orient='records', force_ascii=False, indent=4)
print(f"🚀 Data successfully exported to {OUTPUT_FILE}")